# P-2026-11-15-1 — How big must an ECI change be before a report may say "increased" or "decreased"?

**Question** (from `proposals/2026-11-14.md` #1, queue Q01–Q16 all taken on `main`): use a product bootstrap on the fixed HS4 country panel to estimate, for every country, the probability that its 2019 → 2022 ECI change has the sign we would print. Then propose a threshold, in SD units, below which the report template should print *"essentially unchanged"*.

- **Date:** 2026-11-15 (nightly run)
- **Data:** `data/raw/2019–2023_mega.csv` → HS4 (reporters ≥ $1B, chapters 98/99 dropped, regional aggregates dropped) via `lib/research.repo_hs`; ECI via `lib/eci.eci_from_exports` (RCA ≥ 1, eigenvalues sorted, sign oriented to diversity). Benchmark: `external-data/raw/atlas_hs92_country_year.csv` (Growth Lab ECI).
- **Years:** 2019 → 2022 primary; 2019 → 2021, 2019 → 2023 and the one-year 2021 → 2022 as checks.
- **Run time:** printed in the last code cell.

## TL;DR

- **The pre-stated rule finds no threshold.** At no band τ from 0 to 0.40 SD do ≥ 90% of the countries above it reach ≥ 90% sign confidence (Section C, `tau* = nan` in every window and both bootstraps). Under the paired product bootstrap, the median 2019 → 2022 change on the 112-country fixed HS4 panel is **0.092 SD**, but the median bootstrap SE of that change is **0.159 SD**. Only **9.8%** of countries have p_sign ≥ 0.90. Even for |Δ| ≥ 0.3 SD, only half do.
- **A per-country z rule (|Δ| ≥ 1.645·SE) would call only 7 of 112 countries "changed" for 2019 → 2022**, and all 7 agree with ATLAS's sign (1.000). It would label **all 7 Spring 2026 report countries "essentially unchanged"**, including the UK (+0.205 SD, p_sign 0.942) and Poland (−0.124 SD, p_sign 0.910).
- **The bootstrap is probably too pessimistic, but the band still carries real information.** With the 0.071 SD band from the 11-14 run, ATLAS agrees with our sign on **0.686** of the calls above it vs **0.548** below it (2019 → 2022), and the gap holds in all four windows (Section D). Post hoc, the smallest band giving ≥ 0.80 agreement with ATLAS is 0.255 SD for 2019 → 2022, 0.090 for 2019 → 2021, 0.030 for 2019 → 2023 and 0.100 for 2021 → 2022 (Section F). That spread is too wide to support one universal constant.
- **For the template:** with the 0.071 band, 4 of 7 Spring 2026 countries (Germany, Singapore, UAE, Vietnam) would read "essentially unchanged", and Canada, Poland and the UK keep a direction. There is no data-backed single cut-off yet. My recommendation is to print the change with its bootstrap SE and use the z rule for the word, or else adopt 0.10 SD as a stated convention rather than an estimate. **Will's call** (see ⚠️ in the PR).

## Why this matters for the lab

The 2026-11-14 run (P-2026-11-14-1) showed that moving the reports to oriented HS4 ECI on a fixed panel fixes most of the sign problems. It also showed that several "ECI increased/decreased" calls rest on changes of a few hundredths of an SD, smaller than the typical one-year move. Rule 5 says numbers must trace to notebook output. A direction word that the data can't support is a claim the numbers don't back. This run puts a number on "too small to call", so the template's `[increased/decreased]` placeholder can get a third option, `essentially unchanged`, with a defensible cut-off. It tests **our implementation** (report wording), not the ECI → growth link.

## Method

**Pre-stated before computing anything:**

- **Sample:** the fixed HS4 panel = countries in the HS4 sample in *every* year 2019–2023, and HS4 products present in every year (so one bootstrap draw can be applied to every year). Point ECI is computed on the same panel × products.
- **Change:** Δ_c = ECI_c(2022) − ECI_c(2019), in within-year SD units (ECI is standardised each year).
- **Uncertainty model (primary): paired product bootstrap.** Draw the 1,222 HS4 product columns with replacement, apply the **same** draw to every year, recompute ECI each year, take Δ. B = 500 draws, seed 20261115. Pairing reflects that a report compares the same basket in two years.
- **Primary metric:** sign confidence p_c = share of draws where sign(Δ_b) = sign(Δ_c).
- **Threshold rule (pre-stated):** on a grid τ = 0.000, 0.005, …, 0.400, τ\* is the smallest τ such that **at least 90%** of panel countries with |Δ| ≥ τ have p_c ≥ 0.90. Below τ\*, print "essentially unchanged".
- **Validation (secondary metric):** among panel countries that ATLAS also scores, agreement of our sign with ATLAS's 2019 → 2022 sign for |Δ| ≥ τ\* vs |Δ| < τ\*. If the band is meaningful, agreement should be clearly higher above it.
- **Variants, all reported:** (i) **independent** bootstrap (a fresh product draw for each year, which is a more pessimistic noise model); (ii) a per-country **z rule** |Δ| / SE_boot ≥ 1.645 instead of one fixed τ; (iii) other windows 2019 → 2021, 2019 → 2023, 2021 → 2022; (iv) the 0.071 SD "median one-year move" from P-2026-11-14-1, which is the ad-hoc band that run suggested.
- **Report countries and peers:** the Spring 2026 seven with the peers from the reports (as in the 11-14 run). Merges on ISO3.

In [1]:
import sys, time, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 30)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]
REPORT = ['CAN', 'DEU', 'POL', 'SGP', 'ARE', 'GBR', 'VNM']
PEERS = {'CAN': ['USA', 'AUS', 'GBR'], 'DEU': ['FRA', 'GBR', 'ITA'], 'POL': ['DEU', 'CZE', 'HUN'],
         'SGP': ['KOR', 'JPN', 'DEU'], 'ARE': ['SAU', 'KWT', 'QAT'], 'GBR': ['DEU', 'FRA', 'ITA'],
         'VNM': ['SGP', 'MYS', 'PHL']}
FOCUS = sorted(set(REPORT) | {p for v in PEERS.values() for p in v})
WINDOWS = [(2019, 2022), (2019, 2021), (2019, 2023), (2021, 2022)]
PRIMARY = (2019, 2022)
B, SEED, PSTAR, SHARE = 500, 20261115, 0.90, 0.90
GRID = np.round(np.arange(0, 0.4001, 0.005), 3)
print('lib:', eci.__file__)

lib: /home/user/eci-research/willbetts/nightly/runs/2026-11-15_P-2026-11-15-1-unchanged-band/../../lib/eci.py


### A. Fixed panel and point changes

In [2]:
X = {y: R.repo_hs(y, 4) for y in YEARS}
panel = sorted(set.intersection(*[set(X[y].index) for y in YEARS]))
prods = sorted(set.intersection(*[set(X[y].columns) for y in YEARS]))
XP = {y: X[y].loc[panel, prods] for y in YEARS}
E = pd.DataFrame({y: eci.eci_from_exports(XP[y]) for y in YEARS})
print('per-year HS4 sample sizes:', {y: X[y].shape[0] for y in YEARS})
print('fixed panel:', len(panel), 'countries x', len(prods), 'products;  ECI missing cells:', int(E.isna().sum().sum()))
print('focus countries not in panel:', [c for c in FOCUS if c not in panel] or 'none')
D = {w: (E[w[1]] - E[w[0]]) for w in WINDOWS}
pd.DataFrame({f'{a}->{b}': D[(a, b)].abs().describe() for a, b in WINDOWS}).round(3)

per-year HS4 sample sizes: {2019: 133, 2020: 130, 2021: 136, 2022: 119, 2023: 131}
fixed panel: 112 countries x 1222 products;  ECI missing cells: 0
focus countries not in panel: ['SAU']


,2019->2022,2019->2021,2019->2023,2021->2022
count,112.000,112.000,112.000,112.000
mean,0.176,0.156,0.214,0.117
std,0.264,0.221,0.276,0.134
min,0.000,0.000,0.004,0.001
25%,0.054,0.041,0.060,0.034
50%,0.092,0.090,0.123,0.070
75%,0.182,0.161,0.243,0.134
max,1.697,1.716,1.664,0.762


Takeaway: the panel and the size of typical |Δ| per window (table above). Saudi Arabia is expected to be missing (no 2022 exports in the raw pull).

### B. Paired product bootstrap (primary) and independent bootstrap (variant)

In [3]:
def boot(paired, seed):
    rng = np.random.default_rng(seed); P = len(prods); out = {y: [] for y in YEARS}
    for b in range(B):
        cols = rng.integers(0, P, P)
        for y in YEARS:
            if not paired:
                cols = rng.integers(0, P, P)
            Xb = XP[y].iloc[:, cols]; Xb.columns = range(P)
            out[y].append(eci.eci_from_exports(Xb).reindex(panel))
    return {y: pd.concat(out[y], axis=1) for y in YEARS}   # panel x B

t = time.time(); BP = boot(True, SEED);  print('paired bootstrap: %.0f s' % (time.time() - t))
t = time.time(); BI = boot(False, SEED + 1); print('independent bootstrap: %.0f s' % (time.time() - t))
print('missing country-draws (country lost all RCA>=1 in a draw):', {y: int(BP[y].isna().sum().sum()) for y in YEARS})

def stats_for(BS, w):
    d0 = D[w]; db = BS[w[1]] - BS[w[0]]
    same = np.sign(db).eq(np.sign(d0), axis=0)
    return pd.DataFrame({'delta': d0, 'abs_delta': d0.abs(), 'se_boot': db.std(axis=1),
                         'p_sign': same.mean(axis=1), 'boot_mean': db.mean(axis=1)})
S = {(k, w): stats_for(BS, w) for k, BS in [('paired', BP), ('indep', BI)] for w in WINDOWS}
s = S[('paired', PRIMARY)]
print('PRIMARY paired 2019->2022: median SE_boot = %.3f, share of countries with p_sign >= 0.90 = %.3f'
      % (s.se_boot.median(), (s.p_sign >= PSTAR).mean()))
print('corr(|delta|, SE_boot) =', round(s.abs_delta.corr(s.se_boot), 3))

paired bootstrap: 73 s


independent bootstrap: 72 s
missing country-draws (country lost all RCA>=1 in a draw): {2019: 0, 2020: 0, 2021: 0, 2022: 0, 2023: 0}


PRIMARY paired 2019->2022: median SE_boot = 0.159, share of countries with p_sign >= 0.90 = 0.098
corr(|delta|, SE_boot) = 0.607


Takeaway: the bootstrap SE sets the scale of product-sampling noise in a 3-year ECI change (printed above).

### C. Sign confidence vs |Δ| and the threshold τ\*

In [4]:
def tau_star(st):
    rows = []
    for tau in GRID:
        a = st[st.abs_delta >= tau]
        rows.append({'tau': tau, 'n_above': len(a), 'share_conf': (a.p_sign >= PSTAR).mean() if len(a) else np.nan})
    T = pd.DataFrame(rows)
    ok = T[T.share_conf >= SHARE]
    return (float(ok.tau.min()) if len(ok) else np.nan), T

TAU, TT = {}, {}
for key, st in S.items():
    TAU[key], TT[key] = tau_star(st)
tab = pd.DataFrame({k: {f'{w[0]}->{w[1]}': TAU[(k, w)] for w in WINDOWS} for k in ['paired', 'indep']})
tab.columns = ['tau* paired', 'tau* independent']
TSTAR = TAU[('paired', PRIMARY)]
print('PRIMARY tau* (paired, 2019->2022) =', TSTAR)
tab

PRIMARY tau* (paired, 2019->2022) = nan


,tau* paired,tau* independent
2019->2022,NaN,NaN
2019->2021,NaN,NaN
2019->2023,NaN,NaN
2021->2022,NaN,NaN


In [5]:
T = TT[('paired', PRIMARY)]
print(T[T.tau.isin([0, 0.025, 0.05, 0.071, 0.075, 0.1, 0.125, 0.15, 0.2, 0.25, 0.3])].round(3).to_string(index=False))
# p_sign within |delta| bins
bins = [0, 0.025, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3, 10]
s = S[('paired', PRIMARY)].copy(); s['bin'] = pd.cut(s.abs_delta, bins, right=False)
s.groupby('bin', observed=True).agg(n=('p_sign', 'size'), median_p_sign=('p_sign', 'median'),
                                    share_p90=('p_sign', lambda v: (v >= PSTAR).mean())).round(3)

  tau  n_above  share_conf
0.000      112       0.098
0.025      100       0.110
0.050       86       0.128
0.075       66       0.167
0.100       53       0.208
0.125       41       0.244
0.150       35       0.286
0.200       26       0.385
0.250       19       0.421
0.300       12       0.500


,n,median_p_sign,share_p90
bin,,,
"[0.0, 0.025)",12,0.569,0.000
"[0.025, 0.05)",14,0.559,0.000
"[0.05, 0.075)",20,0.693,0.000
"[0.075, 0.1)",13,0.736,0.000
"[0.1, 0.15)",18,0.781,0.056
"[0.15, 0.2)",9,0.816,0.000
"[0.2, 0.3)",14,0.837,0.286
"[0.3, 10.0)",12,0.877,0.500


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, k, t in zip(axes, ['paired', 'indep'], ['Paired product bootstrap (primary)', 'Independent bootstrap (variant)']):
    st = S[(k, PRIMARY)]
    ax.scatter(st.abs_delta, st.p_sign, s=14, c='#4c72b0', alpha=.7)
    for c in REPORT:
        if c in st.index:
            ax.scatter(st.abs_delta[c], st.p_sign[c], s=40, c='#d62728'); ax.annotate(c, (st.abs_delta[c], st.p_sign[c]), fontsize=8, xytext=(3, -9), textcoords='offset points')
    tau = TAU[(k, PRIMARY)]
    ax.axvline(tau, color='k', ls='--', lw=1, label=f'tau* = {tau}')
    ax.axvline(0.071, color='gray', ls=':', lw=1, label='0.071 (11-14 median 1-yr move)')
    ax.axhline(PSTAR, color='gray', lw=.6)
    ax.set_xscale('symlog', linthresh=0.05); ax.set_xlim(0, 2)
    ax.set_xlabel('|ECI change 2019->2022| (SD units)'); ax.set_ylabel('P(sign as printed)'); ax.set_title(t); ax.legend(fontsize=8, loc='lower right')
fig.tight_layout(); fig.savefig(FIG / 'sign_confidence_vs_delta.png', dpi=100); plt.show()

Takeaway: sign confidence rises with |Δ|; the dashed line is the pre-stated τ\*.

### D. Validation against the Growth Lab's ECI direction

In [7]:
atlas = pd.read_csv('../../external-data/raw/atlas_hs92_country_year.csv')
A = atlas.pivot_table(index='country_iso3_code', columns='year', values='eci')
def agree(w, tau, k='paired'):
    st = S[(k, w)]; ad = (A[w[1]] - A[w[0]]).reindex(st.index)
    m = ad.notna(); st, ad = st[m], ad[m]
    hit = np.sign(st.delta) == np.sign(ad)
    hi = st.abs_delta >= tau
    return {'n_atlas': int(m.sum()), 'n_above': int(hi.sum()), 'agree_above': hit[hi].mean(),
            'n_below': int((~hi).sum()), 'agree_below': hit[~hi].mean(), 'agree_all': hit.mean()}
rows = {}
for w in WINDOWS:
    for lab, tau in [('tau*', TAU[('paired', w)]), ('tau*_primary', TSTAR), ('0.071', 0.071)]:
        rows[(f'{w[0]}->{w[1]}', lab, tau)] = agree(w, tau)
VAL = pd.DataFrame(rows).T; VAL.index.names = ['window', 'band', 'tau']
VAL.round(3)

n_atlas  n_above  agree_above  n_below  agree_below  agree_all
window     band         tau                                                                  
2019->2022 tau*         NaN      112.0      0.0          NaN    112.0        0.634      0.634
           tau*_primary NaN      112.0      0.0          NaN    112.0        0.634      0.634
           0.071        0.071    112.0     70.0        0.686     42.0        0.548      0.634
2019->2021 tau*         NaN      112.0      0.0          NaN    112.0        0.696      0.696
           tau*_primary NaN      112.0      0.0          NaN    112.0        0.696      0.696
           0.071        0.071    112.0     66.0        0.788     46.0        0.565      0.696
2019->2023 tau*         NaN      112.0      0.0          NaN    112.0        0.777      0.777
           tau*_primary NaN      112.0      0.0          NaN    112.0        0.777      0.777
           0.071        0.071    112.0     73.0        0.849     39.0        0.641      0.777
2021->2022 tau*         NaN      112.0      0.0          NaN    112.0        0.679      0.679
           tau*_primary NaN      112.0      0.0          NaN    112.0        0.679      0.679
           0.071        0.071    112.0     56.0        0.750     56.0        0.607      0.679

In [8]:
# z rule variant: |delta| / SE_boot >= 1.645, same ATLAS check
zr = {}
for w in WINDOWS:
    st = S[('paired', w)]; ad = (A[w[1]] - A[w[0]]).reindex(st.index); m = ad.notna()
    z = (st.abs_delta / st.se_boot)[m]; hit = (np.sign(st.delta) == np.sign(ad))[m]; hi = z >= 1.645
    zr[f'{w[0]}->{w[1]}'] = {'n_called': int(hi.sum()), 'agree_called': hit[hi].mean(), 'n_unchanged': int((~hi).sum()), 'agree_unchanged': hit[~hi].mean()}
pd.DataFrame(zr).T.round(3)

,n_called,agree_called,n_unchanged,agree_unchanged
2019->2022,7.0,1.000,105.0,0.610
2019->2021,10.0,0.900,102.0,0.676
2019->2023,24.0,0.958,88.0,0.727
2021->2022,7.0,1.000,105.0,0.657


Takeaway: whether ATLAS agrees more often with the calls above the band than below it (if not, the band is only removing product-sampling noise, not data or method differences).

### E. What the template would print for the Spring 2026 countries and peers

In [9]:
st = S[('paired', PRIMARY)]; sti = S[('indep', PRIMARY)]
ad = (A[PRIMARY[1]] - A[PRIMARY[0]])
def word(d, tau):
    if not np.isfinite(tau): return 'no tau* (rule failed)'
    return 'essentially unchanged' if abs(d) < tau else ('increased' if d > 0 else 'decreased')
rows = []
for c in FOCUS:
    if c not in st.index:
        rows.append({'iso3': c, 'note': 'not in fixed panel'}); continue
    r = st.loc[c]
    rows.append({'iso3': c, 'report': c in REPORT, 'eci_2019': E.loc[c, 2019], 'eci_2022': E.loc[c, 2022], 'delta': r.delta,
                 'se_boot': r.se_boot, 'p_sign': r.p_sign, 'p_sign_indep': sti.loc[c, 'p_sign'],
                 'word_tau*': word(r.delta, TSTAR), 'word_z': word(r.delta, 1.645 * r.se_boot), 'word_0.071': word(r.delta, 0.071),
                 'atlas_delta': ad.get(c, np.nan)})
TAB = pd.DataFrame(rows).set_index('iso3')
TAB.to_csv(FIG / 'focus_bands_2019_2022.csv', float_format='%.4f')
TAB.round(3)

,report,eci_2019,eci_2022,delta,se_boot,p_sign,p_sign_indep,word_tau*,word_z,word_0.071,atlas_delta,note
iso3,,,,,,,,,,,,
ARE,True,-0.078,-0.122,-0.044,0.214,0.436,0.484,no tau* (rule failed),essentially unchanged,essentially unchanged,-0.015,NaN
AUS,False,-0.695,-0.666,0.028,0.205,0.574,0.560,no tau* (rule failed),essentially unchanged,essentially unchanged,-0.095,NaN
CAN,True,0.311,0.392,0.081,0.110,0.594,0.514,no tau* (rule failed),essentially unchanged,increased,-0.049,NaN
CZE,False,1.347,1.330,-0.017,0.117,0.656,0.602,no tau* (rule failed),essentially unchanged,essentially unchanged,-0.017,NaN
DEU,True,1.701,1.675,-0.026,0.127,0.754,0.604,no tau* (rule failed),essentially unchanged,essentially unchanged,-0.024,NaN
FRA,False,0.958,0.961,0.002,0.093,0.550,0.562,no tau* (rule failed),essentially unchanged,essentially unchanged,-0.056,NaN
GBR,True,1.316,1.522,0.205,0.131,0.942,0.798,no tau* (rule failed),essentially unchanged,increased,0.088,NaN
HUN,False,1.034,0.961,-0.073,0.105,0.804,0.664,no tau* (rule failed),essentially unchanged,decreased,0.013,NaN
ITA,False,0.942,0.969,0.028,0.082,0.448,0.484,no tau* (rule failed),essentially unchanged,essentially unchanged,0.035,NaN


In [10]:
print('Report-country words under tau* =', TSTAR)
for c in REPORT:
    if c in TAB.index and isinstance(TAB.loc[c, 'word_tau*'], str):
        r = TAB.loc[c]; print(f"  {c}: delta {r.delta:+.3f} SD, p_sign {r.p_sign:.3f} | tau*: '{r['word_tau*']}' | z rule: '{r.word_z}' | 0.071 band: '{r['word_0.071']}' | ATLAS delta {r.atlas_delta:+.3f}")
for col in ['word_z', 'word_0.071']:
    print(f'report countries "essentially unchanged" under {col}:', int((TAB.loc[REPORT, col] == 'essentially unchanged').sum()), 'of 7')

Report-country words under tau* = nan
  CAN: delta +0.081 SD, p_sign 0.594 | tau*: 'no tau* (rule failed)' | z rule: 'essentially unchanged' | 0.071 band: 'increased' | ATLAS delta -0.049
  DEU: delta -0.026 SD, p_sign 0.754 | tau*: 'no tau* (rule failed)' | z rule: 'essentially unchanged' | 0.071 band: 'essentially unchanged' | ATLAS delta -0.024
  POL: delta -0.124 SD, p_sign 0.910 | tau*: 'no tau* (rule failed)' | z rule: 'essentially unchanged' | 0.071 band: 'decreased' | ATLAS delta -0.030
  SGP: delta +0.025 SD, p_sign 0.588 | tau*: 'no tau* (rule failed)' | z rule: 'essentially unchanged' | 0.071 band: 'essentially unchanged' | ATLAS delta -0.031
  ARE: delta -0.044 SD, p_sign 0.436 | tau*: 'no tau* (rule failed)' | z rule: 'essentially unchanged' | 0.071 band: 'essentially unchanged' | ATLAS delta -0.015
  GBR: delta +0.205 SD, p_sign 0.942 | tau*: 'no tau* (rule failed)' | z rule: 'essentially unchanged' | 0.071 band: 'increased' | ATLAS delta +0.088
  VNM: delta +0.069 SD, p_

### F. Post-hoc (not pre-stated): relaxed bootstrap criteria and an ATLAS-agreement curve

The pre-stated rule found no τ\* (Section C). These cells are **exploratory**, added after seeing that result. Read them as context, not as the pre-registered answer.

In [11]:
rel = {}
for pstar in [0.90, 0.80, 0.75, 0.70]:
    for share in [0.90, 0.75, 0.50]:
        st = S[('paired', PRIMARY)]; best = np.nan
        for tau in GRID:
            a = st[st.abs_delta >= tau]
            if len(a) >= 10 and (a.p_sign >= pstar).mean() >= share: best = tau; break
        rel[(pstar, share)] = best
REL = pd.Series(rel).unstack(); REL.index.name = 'p_sign >='; REL.columns.name = 'share of countries above tau >='
print('smallest tau (2019->2022, paired; needs >= 10 countries above):'); REL

smallest tau (2019->2022, paired; needs >= 10 countries above):


share of countries above tau >=,0.50,0.75,0.90
p_sign >=,,,
0.70,0.000,0.055,0.125
0.75,0.000,0.085,0.165
0.80,0.080,0.255,NaN
0.90,0.265,NaN,NaN


In [12]:
curve = []
for w in WINDOWS:
    st = S[('paired', w)]; ad = (A[w[1]] - A[w[0]]).reindex(st.index); m = ad.notna()
    hit = (np.sign(st.delta) == np.sign(ad))[m]; absd = st.abs_delta[m]
    for tau in GRID:
        hi = absd >= tau
        if hi.sum() >= 10:
            curve.append({'window': f'{w[0]}->{w[1]}', 'tau': tau, 'n_above': int(hi.sum()), 'agree_above': hit[hi].mean(),
                          'agree_below': hit[~hi].mean() if (~hi).sum() else np.nan})
CV = pd.DataFrame(curve)
first80 = CV[CV.agree_above >= 0.80].groupby('window').tau.min()
print('smallest tau with ATLAS sign agreement >= 0.80 above it (>= 10 countries):'); print(first80.reindex([f'{a}->{b}' for a, b in WINDOWS]).to_string())
fig, ax = plt.subplots(figsize=(7, 4))
for wname, g in CV.groupby('window'):
    ax.plot(g.tau, g.agree_above, label=wname)
ax.axvline(0.071, color='gray', ls=':', lw=1, label='0.071')
ax.axhline(0.5, color='k', lw=.5)
ax.set_xlabel('band tau (SD units)'); ax.set_ylabel('share of calls above tau agreeing with ATLAS sign'); ax.legend(fontsize=8)
ax.set_title('Post-hoc: direction agreement with ATLAS above a band')
fig.tight_layout(); fig.savefig(FIG / 'atlas_agreement_vs_band.png', dpi=100); plt.show()
CV[CV.tau.isin([0, 0.05, 0.071, 0.1, 0.15, 0.2])].round(3).pivot(index='tau', columns='window', values='agree_above')

smallest tau with ATLAS sign agreement >= 0.80 above it (>= 10 countries):
window
2019->2022    0.255
2019->2021    0.090
2019->2023    0.030
2021->2022    0.100


window,2019->2021,2019->2022,2019->2023,2021->2022
tau,,,,
0.00,0.696,0.634,0.777,0.679
0.05,0.797,0.709,0.807,0.757
0.10,0.830,0.717,0.881,0.800
0.15,0.886,0.771,0.896,0.840
0.20,0.833,0.731,0.949,0.842


Takeaway (post-hoc): how fast agreement with ATLAS rises as small changes are dropped. This mixes our noise with ATLAS's own noise and data differences.

### Summary numbers

In [13]:
s = S[('paired', PRIMARY)]; v = VAL.xs(('2019->2022', 'tau*'), level=['window', 'band']).iloc[0]
summary = {
  'panel_countries': len(panel), 'products': len(prods), 'B': B,
  'median_abs_delta_2019_2022': round(float(s.abs_delta.median()), 3),
  'median_se_boot_paired': round(float(s.se_boot.median()), 3),
  'median_se_boot_indep': round(float(S[('indep', PRIMARY)].se_boot.median()), 3),
  'share_p_sign_ge_0.90_paired': round(float((s.p_sign >= PSTAR).mean()), 3),
  'tau_star_paired_2019_2022': TSTAR, 'tau_star_indep_2019_2022': TAU[('indep', PRIMARY)],
  'share_countries_below_tau_star': round(float((s.abs_delta < TSTAR).mean()), 3),
  'atlas_agree_above_tau': round(float(v.agree_above), 3), 'atlas_agree_below_tau': round(float(v.agree_below), 3),
  'atlas_n_above': int(v.n_above), 'atlas_n_below': int(v.n_below),
  'z_rule_calls_2019_2022': int(zr['2019->2022']['n_called']), 'z_rule_atlas_agree': round(float(zr['2019->2022']['agree_called']), 3),
  'band_0.071_atlas_agree_above': round(float(VAL.loc[('2019->2022', '0.071', 0.071), 'agree_above']), 3),
  'band_0.071_atlas_agree_below': round(float(VAL.loc[('2019->2022', '0.071', 0.071), 'agree_below']), 3),
}
for k, val in summary.items(): print(f'{k:34s} {val}')
R.save_json({'summary': summary, 'tau_star': {f'{k}:{w[0]}-{w[1]}': TAU[(k, w)] for k, w in TAU}}, FIG / 'summary.json')
print('runtime: %.1f s' % (time.time() - T0))

panel_countries                    112
products                           1222
B                                  500
median_abs_delta_2019_2022         0.092
median_se_boot_paired              0.159
median_se_boot_indep               0.299
share_p_sign_ge_0.90_paired        0.098
tau_star_paired_2019_2022          nan
tau_star_indep_2019_2022           nan
share_countries_below_tau_star     0.0
atlas_agree_above_tau              nan
atlas_agree_below_tau              0.634
atlas_n_above                      0
atlas_n_below                      112
z_rule_calls_2019_2022             7
z_rule_atlas_agree                 1.0
band_0.071_atlas_agree_above       0.686
band_0.071_atlas_agree_below       0.548
runtime: 147.2 s


## Caveats

- **The product bootstrap measures one kind of uncertainty**: which products happen to define the product space. Resampling with replacement duplicates some columns and drops others, which perturbs the PCI weights more than real year-to-year data noise does. It's likely an upper bound on that source. The independent version (median SE 0.299) is more pessimistic still. Neither model covers sample churn (fixed panel by design), exporter misreporting, or HS revision mixing (data request 2026-11-13).
- **ATLAS is a benchmark, not truth.** Its ECI changes are also small (e.g. ATLAS Δ for Germany −0.024, the UAE −0.015). Disagreement on tiny changes is partly ATLAS noise, so the agreement curves understate how accurate our large calls are, and they mix both sources' noise.
- **Short windows, 112 countries, one primary window.** The p_sign bins hold 9–20 countries each. The post-hoc ≥ 0.80 thresholds jump between windows (0.030 → 0.255), which is the sign of a noisy, small-sample estimate.
- **2023 is suspect (D1)**, so the 2019 → 2023 window is a check only. Saudi Arabia is not in the panel (no 2022 exports in the raw pull), so the UAE's peer set is missing one country.
- Point ECIs use the panel × the 1,222 products common to all five years, so values differ slightly from the 11-14 run's per-year H4 (e.g. Canada's 2019 → 2022 Δ is +0.081 here vs −0.022 in H4 there). This is the same sample-churn point that run made.
- Section F was not pre-stated. It is exploratory.

## Helpful to know

- One ECI on the 112 × 1,222 fixed panel takes about 0.03 s. 500 paired draws × 5 years takes 75 s, so the bootstrap is cheap enough to run inside the report notebook for every country.
- **The paired bootstrap halves the noise of a change** (median SE 0.159 vs 0.299 independent). Any uncertainty band for "ECI changed from X to Y" should apply the same product draw to both years.
- SE varies a lot by country (corr(|Δ|, SE) = 0.607). Small Gulf exporters have SE ≈ 0.74 (Kuwait, Qatar), against 0.082–0.127 for Italy, Poland and Germany. A single fixed τ is therefore the wrong shape: it would be too loose for Kuwait and too strict for Italy. That argues for the per-country z rule.
- Kuwait's −0.854 SD drop 2019 → 2022 is large, but its p_sign is only 0.776. Commodity exporters with few RCA products are the least stable ECIs.
- `figures/focus_bands_2019_2022.csv` holds Δ, SE, p_sign and the three candidate words for every report country and peer.

## New research avenues (also in `proposals/2026-11-15.md`)

1. **A less pessimistic noise model.** Repeat Section B with an 80% subsample of products without replacement (rescaled m-out-of-n), and with a country-trade-noise bootstrap (multiplicative log-normal noise on each X_cp, calibrated to the 2019–2022 reporter/ATLAS total ratio spread from Q06). Check whether τ\* exists under either.
2. **Bootstrap the ATLAS HS4 panel the same way** (`external-data/raw/atlas_hs92_product_year_4.csv`) and compare its SEs with ours. That separates "our data is noisy" from "ECI changes are inherently noisy".
3. **Report-template prototype:** a function that prints "ECI rose from X to Y (±SE), a change that is / is not distinguishable from zero", driven by the paired bootstrap, for the 7 Spring 2026 countries. Diff it against the 11-14 claim table.
4. **Use longer windows when they arrive.** Once 2024 data exists, check whether 2019 → 2024 changes clear the z rule more often (signal grows with the window, and paired SE may not).

## Data issues / data needed

No new data problems were found tonight. The existing requests still apply: Saudi Arabia missing in 2022 and the inflated 2023 file (`data-requests/2026-11-12-2023-repull.md`), and the HS-revision mix (`data-requests/2026-11-13-hs-revision-mix.md`). No new `data-requests/` file was written.